# Library

In [49]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# We use gpu if possible (Sam: I have a good gpu) 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Continuous case

## Black & Scholes (Explicit)

Cumulative distribution function of the standard gaussian is computed as:
$$
\Phi(x)=\frac{1}{2}\left(1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right)
$$

In [50]:
class BlackScholes:
    def __init__(self, r, sigma):
        self.r = r
        self.sigma = sigma

    def norm_cdf(self, x):
        return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

    def price(self, S, K, t, T, option_type="call"):
        tau = T - t
        eps = 1e-12

        tau_safe = torch.clamp(tau, min=eps)
        S_safe = torch.clamp(S, min=eps)

        d1 = (torch.log(S_safe / K) + (self.r + 0.5 * self.sigma**2) * tau_safe
             ) / (self.sigma * torch.sqrt(tau_safe))

        d2 = d1 - self.sigma * torch.sqrt(tau_safe)

        if option_type == "call":
            price = ( S_safe * self.norm_cdf(d1) - K * torch.exp(-self.r * tau_safe) * self.norm_cdf(d2))
            payoff = torch.maximum(S - K, torch.zeros_like(S))

        elif option_type == "put":
            price = ( K * torch.exp(-self.r * tau_safe) * self.norm_cdf(-d2) - S_safe * self.norm_cdf(-d1))
            payoff = torch.maximum(K - S, torch.zeros_like(S))

        else:
            raise ValueError("option_type must be 'call' or 'put'")
        
        return torch.where(tau <= eps, payoff, price)

## PINN Class (Under Black & Scholes Framework)

In [51]:
class PINN_BS_Continuous(nn.Module): # I inherit from PyTorch base NN class
    def __init__(self, 
                 K, r, sigma, T,
                 S_min, S_max, t_min, t_max, 
                 hidden_layers,
                 activation_fun = nn.Tanh,
                 loss_weights = None
                 ):
        super().__init__()

        # B&S parameters
        self.K = K
        self.r = r
        self.sigma = sigma
        self.T = T

        # PDE range
        self.S_min = S_min
        self.S_max = S_max
        self.t_min = t_min
        self.t_max = t_max

        # Network Architecture
        self.hidden_layers = hidden_layers
        self.activation_fun = activation_fun
        self.mse = nn.MSELoss()

        if loss_weights is None: 
            loss_weights = {
                "pde": 1.0,
                "terminal": 1.0,
                "boundary_S0": 1.0,
                "boundary_Smax": 1.0,
            }
        self.loss_weights = loss_weights

        layers = []

        layers.append(nn.Linear(2, hidden_layers[0])) # Input layer
        layers.append(activation_fun())

        for i in range(len(hidden_layers) - 1): # Hidden layers
            layers.append(nn.Linear(hidden_layers[i], hidden_layers[i + 1]))
            layers.append(activation_fun())

        layers.append(nn.Linear(hidden_layers[-1], 1))  # Output layer

        self.net = nn.Sequential(*layers)

    def forward(self, S, t):
        """
        Scale the input and return predicted option price V(s,t) from NN
        """
        x = torch.cat([S, t], dim=1) # Input concatenation 

        # We scale the input to roughly [-1,1] for better NN training
        x_scaled = torch.empty_like(x)
        x_scaled[:, 0:1] = 2.0 * (x[:, 0:1] - self.S_min) / (self.S_max - self.S_min) - 1.0
        x_scaled[:, 1:2] = 2.0 * (x[:, 1:2] - self.t_min) / (self.t_max - self.t_min) - 1.0

        return self.net(x_scaled) 
    
    def gradients(self, y, x):
        """
        Compute the derivative of y with respect to x using PyTorch autograd/autodiff.
        """
        return torch.autograd.grad( # compute gradient using autograd
               y, x,                       # of y w.r.t x
               grad_outputs=torch.ones_like(y), # weight of the sum are 1 dimension as y 
               create_graph=True,         # create derivative chain for second order derivative
               retain_graph=True          # keep such chain
               )[0]                       # extracts the actual gradient tensor 
    
    def bs_pde_residual(self, S, t):
        """
        It compute the Black-Scholes PDE residual in (S,t) defined as:
            V_tau - 0.5*sigma^2*S^2*V_SS - r*S*V_S + r*V = 0
        """
        S = S.clone().detach().requires_grad_(True) # For safety we copy, remove any attached graph a
        t = t.clone().detach().requires_grad_(True) # and we state that V will be differentiable with S, t 

        V = self.forward(S, t)

        V_t = self.gradients(V, t)
        V_S = self.gradients(V, S)
        V_SS = self.gradients(V_S, S)

        residual = V_t + 0.5 * self.sigma**2 * S**2 * V_SS + self.r * S * V_S - self.r * V

        return residual
    
    def sample_training_points(self, N_f, N_terminal, N_b):
        """
        Sample training points for the PINN.

        N_f : Number of interior collocation points for the PDE residual.
        N_terminal : Number of terminal-condition points at t = T.
        N_b : Number of points for each spatial boundary.
        """
        device = next(self.parameters()).device
        eps = 1e-6

        # Interior points
        S_f = torch.rand(N_f, 1, device=device) * (self.S_max - eps) + eps
        t_f = torch.rand(N_f, 1, device=device) * (self.t_max - self.t_min) + self.t_min

        # Terminal condition points: t = T
        S_T = torch.rand(N_terminal, 1, device=device) * (self.S_max - self.S_min) + self.S_min
        t_T = torch.full((N_terminal, 1), self.T, device=device)

        # Boundary S = 0
        S_b0 = torch.zeros(N_b, 1, device=device)
        t_b0 = torch.rand(N_b, 1, device=device) * (self.t_max - self.t_min) + self.t_min

        # Boundary S = S_max
        S_b1 = torch.full((N_b, 1), self.S_max, device=device)
        t_b1 = torch.rand(N_b, 1, device=device) * (self.t_max - self.t_min) + self.t_min

        return S_f, t_f, S_T, t_T, S_b0, t_b0, S_b1, t_b1
    

    def compute_loss_from_data(self, data):
        """
        Compute the total PINN loss on a fixed dataset (required for L-BFGS).
        """
        S_f, t_f, S_T, t_T, S_b0, t_b0, S_b1, t_b1 = data

        # PDE residual loss
        residual_f = self.bs_pde_residual(S_f, t_f)
        loss_pde = self.mse(residual_f, torch.zeros_like(residual_f))

        # Terminal condition: V(S, T) = max(S - K, 0)
        V_pred_T = self.forward(S_T, t_T)
        V_true_T = torch.maximum(S_T - self.K, torch.zeros_like(S_T))
        loss_terminal = self.mse(V_pred_T, V_true_T)

        # Boundary condition at S = 0: V(0, t) = 0
        V_pred_b0 = self.forward(S_b0, t_b0)
        V_true_b0 = torch.zeros_like(S_b0)
        loss_b0 = self.mse(V_pred_b0, V_true_b0)

        # Boundary condition at S = S_max: V(S_max, t) = S_max - K exp(-r(T-t))
        V_pred_b1 = self.forward(S_b1, t_b1)
        V_true_b1 = S_b1 - self.K * torch.exp(-self.r * (self.T - t_b1))
        loss_b1 = self.mse(V_pred_b1, V_true_b1)

        # Total weighted loss
        loss = (
            self.loss_weights["pde"] * loss_pde
            + self.loss_weights["terminal"] * loss_terminal
            + self.loss_weights["boundary_S0"] * loss_b0
            + self.loss_weights["boundary_Smax"] * loss_b1
        )

        return loss, {
            "loss_pde": loss_pde.item(),
            "loss_terminal": loss_terminal.item(),
            "loss_b0": loss_b0.item(),
            "loss_b1": loss_b1.item(),
            }

    def compute_loss(self, N_f, N_terminal, N_b):
        """
        Compute the total PINN loss, resample first than compute. 
        """
        data = self.sample_training_points(N_f, N_terminal, N_b)
        return self.compute_loss_from_data(data)

## Initialization

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Black-Scholes parameters
K = 100.0        
r = 0.05        
sigma = 0.20    
T = 1.0

# Ranges
S_min = 0.0
S_max = 150.0
t_min = 0.0
t_max = T

# Training sample sizes
N_f = 10000        # internal points
N_terminal = 1000    # payoff points at tau=0
N_b = 1000         # boundary points

# Network hyperparameters
hidden_layers = [64, 64, 64, 64]
activation_fun = nn.Tanh
mse = nn.MSELoss()

lr = 1e-4
epochs = 5000

# Class Initialization
bs = BlackScholes(r, sigma)
model = PINN_BS_Continuous(K, r, sigma, T,
                           S_min, S_max, t_min, t_max,
                           hidden_layers,
                           activation_fun).to(device)

## Training + Optimization

In [53]:
optimizer = optim.Adam(model.parameters(), lr=lr) # I use Adam since the only one i knwo

for epoch in range(1, epochs + 1):
    optimizer.zero_grad()
    loss, parts = model.compute_loss(N_f, N_terminal, N_b)
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:5d} | "
            f"Total {loss.item():.6e} | "
            f"PDE {parts['loss_pde']:.3e} | "
            f"Terminal {parts['loss_terminal']:.3e} | "
            f"S=0 {parts['loss_b0']:.3e} | "
            f"S=max {parts['loss_b1']:.3e}"
        )

fixed_data = model.sample_training_points(N_f, N_terminal, N_b)

optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=500,
    max_eval=500,
    history_size=50,
    line_search_fn="strong_wolfe"
)

def closure():
    optimizer_lbfgs.zero_grad()
    loss, _ = model.compute_loss_from_data(fixed_data)
    loss.backward()
    return loss

optimizer_lbfgs.step(closure)

Epoch     1 | Total 3.067852e+03 | PDE 8.470e-03 | Terminal 2.954e+02 | S=0 1.710e-02 | S=max 2.772e+03
Epoch   500 | Total 2.228500e+03 | PDE 8.355e-01 | Terminal 1.984e+02 | S=0 3.288e-01 | S=max 2.029e+03
Epoch  1000 | Total 1.863794e+03 | PDE 2.220e+00 | Terminal 1.361e+02 | S=0 4.015e-02 | S=max 1.725e+03
Epoch  1500 | Total 1.574595e+03 | PDE 2.524e+00 | Terminal 1.062e+02 | S=0 4.255e-02 | S=max 1.466e+03
Epoch  2000 | Total 1.317702e+03 | PDE 2.374e+00 | Terminal 7.226e+01 | S=0 1.596e-02 | S=max 1.243e+03
Epoch  2500 | Total 1.109938e+03 | PDE 2.604e+00 | Terminal 5.405e+01 | S=0 8.801e-03 | S=max 1.053e+03
Epoch  3000 | Total 9.285882e+02 | PDE 3.096e+00 | Terminal 4.849e+01 | S=0 1.025e-02 | S=max 8.770e+02
Epoch  3500 | Total 7.573277e+02 | PDE 3.559e+00 | Terminal 3.381e+01 | S=0 2.819e-03 | S=max 7.200e+02
Epoch  4000 | Total 6.219895e+02 | PDE 4.150e+00 | Terminal 2.848e+01 | S=0 3.170e-03 | S=max 5.894e+02
Epoch  4500 | Total 4.955183e+02 | PDE 4.775e+00 | Terminal 1.75

tensor(389.3090, device='cuda:0', grad_fn=<AddBackward0>)

## Evaluation

In [ ]:
model.eval()

with torch.no_grad():
    nS = 200
    nT = 100

    S_grid = torch.linspace(S_min, S_max, nS, device=device).view(-1, 1)
    t_grid = torch.linspace(t_min, t_max, nT, device=device).view(-1, 1)

    SS, TT = torch.meshgrid(
        S_grid.squeeze(),
        t_grid.squeeze(),
        indexing="ij"
    )

    S_test = SS.reshape(-1, 1)
    t_test = TT.reshape(-1, 1)

    V_pred = model(S_test, t_test)
    V_exact = bs.price(S=S_test, K=K, t=t_test, T=T, option_type="call")

    abs_error = torch.abs(V_pred - V_exact)
    rel_l2 = torch.norm(V_pred - V_exact) / torch.norm(V_exact)
    rmse = torch.sqrt(torch.mean((V_pred - V_exact) ** 2))
    max_abs_error = torch.max(abs_error)

    print("Evaluation on test grid:")
    print(f"Relative L2 error : {rel_l2.item():.6e}")
    print(f"RMSE              : {rmse.item():.6e}")
    print(f"Max abs error     : {max_abs_error.item():.6e}")


Evaluation on test grid:
Relative L2 error : 2.673633e-03
RMSE              : 4.900005e-02
Max abs error     : 1.106208e+00


## Single Case

In [55]:
with torch.no_grad():
    S_example = torch.tensor([[100.0]], device=device)
    t_example = torch.tensor([[0.5]], device=device)

    pred = model(S_example, t_example).item()
    exact = bs.price(S_example, K=K, t=t_example, T=T, option_type="call").item()

    print("\nExample at S=100, t=0.5:")
    print(f"PINN  price: {pred:.6f}")
    print(f"Exact price: {exact:.6f}")
    print(f"Abs error  : {abs(pred - exact):.6f}")


Example at S=100, t=0.5:
PINN  price: 6.919711
Exact price: 6.888725
Abs error  : 0.030985


# Discrete Case